### Universidade Federal da Bahia
### Escola Politécnica
### Programa de Pós-Graduação em Engenharia Industrial
### Discente: Ghabriel Anton Gomes de Sá
### Orientadores: Marcelo Embiruçu e Cristiano Fontes

<center><h1>UMA ABORDAGEM COMBINANDO UMA ESTIMATIVA DE PESOS INICIAIS BASEADA EM LINEARIZAÇÃO E ALGORITMO CONSTRUTIVISTA PARA A CRIAÇÃO DE REDES NEURAIS DE PROPAGAÇÃO DIRETA COM UMA ÚNICA CAMADA OCULTA APLICADAS À IDENTIFIÇÃO DE MODELOS COM MÚLTIPLAS ENTRADAS E ÚNICA SAÍDA EM PROBLEMAS DE REGRESSÃO COM POSSIBILIDADE DE RESTRIÇÕES NOS SINAIS DOS GANHOS</center></h1>

#### <p style='text-align: justify;'>O Método para a obtenção dos pesos iniciais propõe que: (i) seja realizada a linearização, via série de Taylor, do modelo não-linear a ser utilizado para a modelagem do conjunto de dados (o modelo consiste em uma rede neural com p entradas e um neurônio na camada de saída), onde o ponto de equilíbrio para a linearização é a média das suas respectivas variáveis (x,y); (ii) seja realizada a Regressão Linear Múltipla, via Método dos Mínimos Quadrados, do conjunto de dados a ser modelado; (iii) através da comparação matemática entre (i) e (ii), obtenha-se os pesos que conectam os neurônios de entrada ao primeiro neurônio oculto da rede. </p>

#### <p style='text-align: justify;'>Em seguida, a rede continua a aumentar o número de neurônios ocultos a partir de uma abordagem construtivista. Toda a rede é treinada a cada adição de um novo neurônio oculto, sendo que os pesos obtidos no final de um treinamento são utilizados como pesos iniciais para o treinamento subsequente. Os pesos iniciais do neurônio oculto a ser adicionado são obtidos a partir do Método de Xavier, enquanto biases são inicializados com valor nulo. Comparou-se o método proposto com outros dois métodos (RIXM e ELM). </p>

## Importação de pacotes

In [ ]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error as mse

import statsmodels.api as sm
from scipy.optimize import fsolve
from scipy.optimize import minimize

import numpy as np
import math
from scipy.linalg import pinv

import time

import matplotlib.pyplot as plt
%matplotlib inline

## Fixação de seed
A fixação de seed garante a reprodutibilidade dos resultados.

In [ ]:
np.random.seed(0)

## Importação do banco de dados
<p style='text-align: justify;'>Para que a importação do banco de dados seja feita da maneira correta, é necessário que o banco de dados a ser importado esteja de acordo com o padrão apresentado a seguir e também que ele esteja localizado na mesma pasta que esse código. A princípio, esse código está implementado em um arquivo .ipynb.</p>

O banco de dados deve apresentar, obrigatoriamente, as seguintes características:

-Deve ser um arquivo do tipo .csv;

-Os dados devem estar delimitados por ponto e vírgula (;), ou seja, devem estar presentes em diferentes células da planilha Excel;

-As variáveis de entradas são todas as colunas, exceto a última (que trata-se da variável de saída);

-Não deveve possuir cabeçalho (primeira linha contendo nome dos atributos).

In [ ]:
# Solicita nome do arquivo (.csv) que armazena o conjunto de dados
filename = input("Digite o nome do arquivo .csv que contém o banco de dados: ")

# Realiza a importação do conjunto de dados em formato de DataFrame
try:
    data = pd.read_csv(filename + '.csv', header=None, sep=';')
    print("Dados importados com sucesso. Visualização das primeiras linhas do dataframe:")
    print(data.head())
except FileNotFoundError:
    print(f"Erro: O arquivo '{filename}.csv' não foi encontrado. Verifique o nome e a localização do arquivo.")

### Coleta dos sinais de ganhos esperados

In [ ]:
def get_expected_signals(n_inputs):
    """
    Solicita ao usuário os sinais esperados para o ganho de cada variável de entrada.
    
    n_inputs: Número de variáveis de entrada.
    
    Retorna:
    Uma lista contendo os sinais esperados para o ganho de cada variável de entrada.
    """
    expected_signals = []
    print("Informe os sinais esperados dos ganhos variáveis de entrada.")
    print("Digite 1 para um ganho positivo, -1 para um ganho negativo e 0 caso o ganho seja desconhecido:")
    for i in range(n_inputs):
        while True:
            signal = input(f"Sinal esperado para o ganho da variável {i+1}: ")
            if signal in ["1", "-1", "0"]:
                expected_signals.append(int(signal))
                break
            else:
                print("Entrada inválida. Por favor, digite 1, -1, ou 0.")
    return expected_signals

In [ ]:
# Atribui valor à variavel que reflete o número de atributos do conjunto de dados (número de colunas dos dados de entrada):
n_inputs = data.shape[1]-1

# Solicita ao usuário os sinais esperados para o ganho de cada variável de entrada
expected_signals = get_expected_signals(n_inputs)

# Imprime o vetor dos sinais de ganhos em tela
print("Sinais de ganhos esperados:", expected_signals)

## Pré-processamento de dados

#### • Obtendo primeiras informações sobre o conjunto de dados:

In [ ]:
# Imprime cinco primeiras linhas do conjunto de dados
print("Cinco primeiras linhas do conjunto de dados:")
print(data.head())

In [ ]:
# Imprime informações sobre o conjunto de dados
print("\nInformações sobre o conjunto de dados:")
print(data.info())

In [ ]:
# Imprime uma breve descrição estatística dos dados
print("\nDescrição estatística dos dados:")
print(data.describe())

#### • Lidando com os dados faltantes:

In [ ]:
# Verifica a existência de dados faltantes
print("\nVerificação de dados faltantes:")
print(data.isna().sum())

In [ ]:
# Deleta dados faltantes:
data.dropna(inplace=True)

In [ ]:
# Missing values também podem estar representadas através do número 0. Por conta disso, deve-se verificar se os zeros são missing values ou se são
# inerentes ao atributo:
num_zeros = (data == 0).sum()
print('Quantidade de valores nulos:\n', num_zeros)

#### • Reduzindo a escala dos dados:
Essa etapa garante que os dados estejam entre 0 e 1.

In [ ]:
# Cria uma array a partir dos dados
z = data.values

# Normaliza os dados
min_max_scaler = MinMaxScaler()
data_normalized = min_max_scaler.fit_transform(z)
print("\nDados normalizados (primeiras linhas):\n", data_normalized[:5])

#### • Delimitação de entrada e saída:

In [ ]:
# Delimitação de entradas (todas as colunas, exceto a última) e saída (última coluna)
inputs_all = data_normalized[:, :-1]
targets_all = data_normalized[:, -1]

In [ ]:
# Exibe entrada e saída e número de atributos (variáveis de entrada)
print("\nVetores de Entrada (primeiras linhas):\n", inputs_all[:5])
print("\nValores de Saída (primeiras linhas):\n", targets_all[:5])
print("\nNúmero de atributos (entradas):", n_inputs)

#### • Divisão do conjunto de dados em treinamento e teste

In [ ]:
# Embaralha e divide os dados utilizando train_test_split
train_inputs, test_inputs, train_targets, test_targets = train_test_split(
    inputs_all, targets_all, test_size=0.2, random_state=42)  # random_state fixado garante reprodutibilidade dos experimentos

In [ ]:
# Exibe o formato dos conjuntos de treinamento e teste
print("\nFormato dos dados de treinamento e teste:")
print("train_inputs:", train_inputs.shape)
print("test_inputs:", test_inputs.shape)
print("train_targets:", train_targets.shape)
print("test_targets:", test_targets.shape)

## Método 1 (WILCAR)
Modelo SFNN, treinado via Algoritmo de Retropropagação, considerando os pesos iniciais (e bias) do primeiro neurônio oculto obtidos a partir de Linearização e Análise via Série de Taylor e pesos dos demais neurônios ocultos, bem como do neurônio de saída obtidos a partir de uma inicialização randômica via Método de Xavier, preservando os pesos dos neurônios já aprendidos e utilizando-os como estimativas iniciais para o próximo treinamento.

### Obtenção dos Pesos Iniciais
Obtenção dos pesos iniciais entre os neurônios de entrada e o primeiro neurônio oculto.

In [ ]:
# Adiciona a coluna das constantes
inputs_all_cte = sm.add_constant(inputs_all, prepend=True)
res = sm.OLS(targets_all, inputs_all_cte).fit()  # Cria e ajusta o modelo
print(res.summary())  # Imprime os resultados

In [ ]:
# Definindo constantes
x_M = [np.mean(inputs_all[:, i]) for i in range(n_inputs)]
k = [res.params[i + 1] for i in range(n_inputs)]
b = res.params[0]

In [ ]:
# Definindo chutes iniciais para beta e wi como valores aleatórios entre 0 e 1:
x0 = np.random.rand(n_inputs+1, 1)

In [ ]:
# Definindo função sigmoide e sua derivada:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def dsigmoid(x):
    z = sigmoid(x)
    return z * (1 - z)

In [ ]:
# Inicializando variáveis wi e beta:
w = [0 for i in range(n_inputs)]
beta = 0

In [ ]:
# Definindo sistema de equações a ser resolvido:
def f(x):
    w, beta = x[:-1], x[-1]
    f = [sigmoid(np.dot(w, x_M)+beta)-(sigmoid(np.dot(w, x_M)+beta) -
                                       (sigmoid(np.dot(w, x_M)+beta))**2)*np.dot(w, x_M)-b]
    for i in range(n_inputs):
        f.append((sigmoid(np.dot(w, x_M)+beta) -
                 (sigmoid(np.dot(w, x_M)+beta))**2)*w[i]-k[i])
    return f

In [ ]:
# Resolvendo o sistema e imprimindo resultados (pesos e bias iniciais):
result = fsolve(f, x0)
print("Pesos e bias iniciais:", result)

### Avaliação dos Sinais dos Ganhos do Modelo Inicial
O modelo inicial corresponde a um Perceptron.

In [ ]:
def calculate_and_compare_gains(adjusted_weights, x_base, expected_signals, n_inputs):
    """
    Calcula os sinais dos ganhos com base nos pesos ajustados e os compara com os sinais esperados.

    Parâmetros:
    - adjusted_weights: array-like
        Pesos ajustados, incluindo os pesos dos neurônios de entrada e o bias.
    - x_base: array-like
        Vetor de entrada base utilizado para calcular os ganhos.
    - expected_signals: list
        Lista contendo os sinais esperados para os ganhos de cada variável de entrada.
    - n_inputs: int
        Número de variáveis de entrada.

    Retorna:
    - None
        A função imprime os resultados da comparação entre os ganhos calculados e os sinais esperados.
    """
    w, beta = adjusted_weights[:-1], adjusted_weights[-1]
    y_base = sigmoid(np.dot(w, x_base) + beta)
    
    calculated_signals = []
    for i in range(n_inputs):
        x_var = x_base.copy()
        x_var[i] += 0.1
        new_y = sigmoid(np.dot(w, x_var) + beta)
        gain_signal = np.sign((new_y - y_base) / 0.1)
        calculated_signals.append(gain_signal)

    # Comparando os sinais dos ganhos calculados com os esperados
    for i, (calculated, expected) in enumerate(zip(calculated_signals, expected_signals)):
        print(f"Variável {i + 1}: Ganho calculado = {calculated}, Ganho esperado = {expected}")
        if expected != 0 and calculated != expected:
            print(f"  -> Divergência detectada na variável {i + 1}.")
        elif expected == 0:
            print(f"  -> Ganho desconhecido esperado; ganho calculado foi {calculated}.")
        else:
            print(f"  -> Correspondência encontrada.")

In [ ]:
# Checagem dos valores dos ganhos associados aos pesos iniciais (utiliza-se como referência o primeiro registro do conjunto de testes)
x_base = test_inputs[0].reshape(-1, 1)
calculate_and_compare_gains(result, x_base, expected_signals, n_inputs)

### Definição da Classe do Método 1

In [ ]:
class Method1:
    """
    Classe para o Método 1 (WILCAR) que treina uma rede neural feedforward com uma única camada oculta.
    
    Parâmetros:
    - x: array-like
        Dados de entrada.
    - y: array-like
        Dados de saída.
    - n: int
        Número de neurônios na camada oculta.
    - previous_model: Method1, opcional
        Modelo anterior para expandir a rede neural.
    - initial_weights: array-like, opcional
        Pesos iniciais fornecidos para o primeiro neurônio oculto.
    """
    
    def __init__(self, x, y, n, previous_model=None, initial_weights=None):
        self.x, self.y = x, y.reshape(1, -1)
        self.n = n
        self.param = self.initialize_parameters(n, initial_weights) if previous_model is None else self.expand_network(previous_model, n)
        self.dims = [x.shape[0], n, 1]
        self.lr = 0.1  # Taxa de aprendizado
        self.sam = y.shape[0]  # Número de amostras

    def initialize_parameters(self, n, initial_weights):
        """
        Inicializa os parâmetros (pesos e biases) da rede neural.
        
        Parâmetros:
        - n: int
            Número de neurônios na camada oculta.
        - initial_weights: array-like, opcional
            Pesos iniciais fornecidos para o primeiro neurônio oculto.
        
        Retorna:
        - param: dict
            Dicionário contendo os pesos e biases iniciais.
        """
        param = {'W1': np.random.randn(n, self.x.shape[0]) * np.sqrt(1. / self.x.shape[0]),
                 'b1': np.zeros((n, 1)),
                 'W2': np.random.randn(1, n) * np.sqrt(1. / n),
                 'b2': np.zeros((1, 1))}
        if initial_weights is not None:
            param['W1'][0, :] = initial_weights[:-1]
            param['b1'][0, :] = initial_weights[-1]
        return param

    def expand_network(self, previous_model, n):
        """
        Expande a rede neural existente com mais neurônios na camada oculta.
        
        Parâmetros:
        - previous_model: Method1
            Modelo anterior para expandir.
        - n: int
            Número de neurônios na nova camada oculta.
        
        Retorna:
        - new_param: dict
            Dicionário contendo os novos pesos e biases expandidos.
        """
        new_param = previous_model.param
        if n > previous_model.n:
            additional_weights = np.random.randn(1, self.x.shape[0]) * np.sqrt(1. / self.x.shape[0])
            new_param['W1'] = np.vstack([new_param['W1'], additional_weights])
            new_param['b1'] = np.vstack([new_param['b1'], np.zeros((1, 1))])
            new_param['W2'] = np.hstack([new_param['W2'], np.random.randn(1, 1) * np.sqrt(1. / n)])
        return new_param

    def sigmoid(self, Z):
        """
        Função de ativação sigmoide.
        
        Parâmetros:
        - Z: array-like
            Entrada para a função sigmoide.
        
        Retorna:
        - array-like
            Saída da função sigmoide.
        """
        return 1 / (1 + np.exp(-Z))

    def forward(self):
        """
        Propagação para frente (forward propagation) através da rede neural.
        
        Retorna:
        - array-like
            Saída da rede neural.
        """
        Z1 = self.param['W1'].dot(self.x) + self.param['b1']
        A1 = self.sigmoid(Z1)
        Z2 = self.param['W2'].dot(A1) + self.param['b2']
        return self.sigmoid(Z2)

    def backward(self, Yh):
        """
        Propagação para trás (backward propagation) para atualizar os pesos e biases.
        
        Parâmetros:
        - Yh: array-like
            Saída predita pela rede neural.
        """
        m = self.y.shape[1]
        dZ2 = Yh - self.y
        dW2 = dZ2.dot(self.sigmoid(self.param['W1'].dot(self.x) + self.param['b1']).T) / m
        db2 = np.sum(dZ2, axis=1, keepdims=True) / m
        dZ1 = self.param['W2'].T.dot(dZ2) * self.sigmoid(self.param['W1'].dot(self.x) + self.param['b1']) * (1 - self.sigmoid(self.param['W1'].dot(self.x) + self.param['b1']))
        dW1 = dZ1.dot(self.x.T) / m
        db1 = np.sum(dZ1, axis=1, keepdims=True) / m
        self.param['W1'] -= self.lr * dW1
        self.param['b1'] -= self.lr * db1
        self.param['W2'] -= self.lr * dW2
        self.param['b2'] -= self.lr * db2

    def train_test(self, iter=1501):
        """
        Treina a rede neural e avalia o desempenho nos dados de teste.
        
        Parâmetros:
        - iter: int, opcional
            Número de iterações de treinamento (padrão: 1501).
        
        Retorna:
        - mse_train: float
            Erro quadrático médio no conjunto de treinamento.
        - r2_train: float
            Coeficiente de determinação R2 no conjunto de treinamento.
        - mse_test: float
            Erro quadrático médio no conjunto de teste.
        - r2_test: float
            Coeficiente de determinação R2 no conjunto de teste.
        """
        for i in range(iter):
            Yh = self.forward()
            self.backward(Yh)
            
        Yh_train = self.forward()
        mse_train = mse(self.y.flatten(), Yh_train.flatten())
        r2_train = r2_score(self.y.flatten(), Yh_train.flatten())
        Yh_test = self.predict(test_inputs.T)
        mse_test = mse(test_targets.T, Yh_test.flatten())
        r2_test = r2_score(test_targets.T, Yh_test.flatten())
        return mse_train, r2_train, mse_test, r2_test

    def predict(self, x_new):
        """
        Faz previsões com base em novos dados de entrada.
        
        Parâmetros:
        - x_new: array-like
            Novos dados de entrada para fazer previsões.
        
        Retorna:
        - array-like
            Previsões da rede neural.
        """
        Z1 = np.dot(self.param['W1'], x_new) + self.param['b1']
        A1 = self.sigmoid(Z1)
        Z2 = np.dot(self.param['W2'], A1) + self.param['b2']
        A2 = self.sigmoid(Z2)
        return A2


### Implementação do Método 1

In [ ]:
import time

# Inicialização de variáveis para rastrear métricas de desempenho
mse_train_1 = []
r2_train_1 = []
mse_test_1 = []
r2_test_1 = []

# Inicialização de variáveis para rastrear o melhor modelo
best_r2_test_1 = -np.inf  # Inicializa com infinito negativo para garantir que qualquer modelo seja melhor que este inicialmente
best_model_1 = None  # Para guardar o melhor modelo
best_n_1 = 0  # Para guardar o número de neurônios do melhor modelo

# Marca o início da contagem do tempo
ini = time.time()

# Loop para treinar modelos com diferentes números de neurônios na camada oculta
for n in range(1, 251):
    if n == 1:
        model = Method1(train_inputs.T, train_targets.T, n, initial_weights=result)
    else:
        model = Method1(train_inputs.T, train_targets.T, n, previous_model=previous_model)
    
    mse_train, r2_train, mse_test, r2_test = model.train_test()
    
    mse_train_1.append(mse_train)
    r2_train_1.append(r2_train)
    mse_test_1.append(mse_test)
    r2_test_1.append(r2_test)
    
    print(f"Nós Ocultos: {n} - MSE Treino = {mse_train}, R2 Treino = {r2_train}, MSE Teste = {mse_test}, R2 Teste = {r2_test}")
    
    # Atualiza o melhor modelo se o atual for melhor que o registrado anteriormente
    if r2_test > best_r2_test_1:
        best_r2_test_1 = r2_test
        best_model_1 = model  # Salva o modelo atual como o melhor modelo
        best_n_1 = n  # Salva o número de neurônios do melhor modelo
    
    # Verifica os ganhos do modelo atual
    x_base = test_inputs.T
    delta = 0.1
    ganhos_calculados = []
    for i in range(n_inputs):
        x_perturbed = np.copy(x_base)
        x_perturbed[i, :] += delta  # Perturba cada variável de entrada individualmente
        y_base = model.predict(x_base)
        y_perturbed = model.predict(x_perturbed)
        ganho_i = (y_perturbed - y_base) / delta
        ganhos_calculados.append(np.mean(ganho_i))  # Tira a média de todos os ganhos calculados, por variável

    # Compara os ganhos calculados com os sinais de ganhos esperados
    for i, (ganho_calculado, sinal_esperado) in enumerate(zip(ganhos_calculados, expected_signals)):
        sinal_calculado = np.sign(ganho_calculado)
        
        if sinal_esperado == 0:
            print(f"Modelo {n}, Variável {i+1}: Ganho calculado = {sinal_calculado}, Ganho esperado = Desconhecido")
        elif sinal_calculado == sinal_esperado:
            print(f"Modelo {n}, Variável {i+1}: Ganho calculado = {sinal_calculado}, Ganho esperado = {sinal_esperado} [Conforme]")
        else:
            print(f"Modelo {n}, Variável {i+1}: Ganho calculado = {sinal_calculado}, Ganho esperado = {sinal_esperado} [Divergente]")

    # Salva o modelo atual para a próxima iteração
    previous_model = model

# Marca o fim da contagem do tempo
end = time.time()
print(f"Método concluído em {end - ini} segundos.")
print(f"Melhor Modelo: {best_n_1} Neurônios, R2 Teste = {best_r2_test_1}")

### Análise da Performance do Método 1

In [ ]:
# Define o conjunto de teste como referência para o cálculo dos ganhos
x_base = test_inputs.T

# Definindo delta (variação nas entradas)
delta = 0.1

# Calcula os ganhos para cada variável de entrada
ganhos_calculados = []
for i in range(n_inputs):
    x_perturbed = np.copy(x_base)
    x_perturbed[i, :] += delta  # Perturba cada variável de entrada individualmente
    y_base = model.predict(x_base)
    y_perturbed = best_model_1.predict(x_perturbed)
    ganho_i = (y_perturbed - y_base) / delta
    ganhos_calculados.append(np.mean(ganho_i))  # Tira a média de todos os ganhos calculados, por variável

print('ANÁLISE DE GANHOS - MELHOR MODELO - WILCAR')

# Compara os ganhos calculados com os sinais de ganhos esperados
for i, (ganho_calculado, sinal_esperado) in enumerate(zip(ganhos_calculados, expected_signals)):
    sinal_calculado = np.sign(ganho_calculado)
    
    if sinal_esperado == 0:
        print(f"Variável {i+1}: Ganho calculado = {sinal_calculado}, Ganho esperado = Desconhecido")
    elif sinal_calculado == sinal_esperado:
        print(f"Variável {i+1}: Ganho calculado = {sinal_calculado}, Ganho esperado = {sinal_esperado} [Conforme]")
    else:
        print(f"Variável {i+1}: Ganho calculado = {sinal_calculado}, Ganho esperado = {sinal_esperado} [Divergente]")

In [ ]:
# Imprime o número ótimo de neurônios ocultos e o melhor R2 de teste
print("Número ótimo de neurônios:", best_n_1)
print("Melhor R2 de Teste:", best_r2_test_1)

In [ ]:
# Criação e exibição de Gráfico do MSE de treino
plt.plot(mse_train_1)
plt.ylabel('MSE de Treino')
plt.xlabel('Nós Ocultos')
plt.legend(['WILCAR'])
plt.show()

In [ ]:
# Criação e exibição de Gráfico do R2 de treino
# plt.plot(r2_train_1)
# plt.ylabel('R2 de Treino')
# plt.xlabel('Nós Ocultos')
# plt.legend(['WILCAR'])
# plt.show()

In [ ]:
# Criação e exibição de Gráfico do MSE de teste
plt.plot(mse_test_1)
plt.ylabel('MSE de Teste')
plt.xlabel('Nós Ocultos')
plt.legend(['WILCAR'])
plt.show()

In [ ]:
# Criação e exibição de Gráfico do R2 de teste
plt.plot(r2_test_1)
plt.ylabel('R2 de Teste')
plt.xlabel('Nós Ocultos')
plt.legend(['WILCAR'])
plt.show()

### Salvando Métricas do Método 1 (WILCAR)

In [ ]:
# Função para salvar métricas
def save_metrics(metrics, experiment_name):
    """
    Salva as métricas em um arquivo CSV.

    Parâmetros:
    - metrics: dict
        Dicionário contendo as métricas a serem salvas.
    - experiment_name: str
        Nome do experimento para nomear o arquivo CSV.
    """
    df = pd.DataFrame(metrics)
    df.to_csv(f"{filename}_{experiment_name}.csv", index=False)

In [ ]:
# Dicionário contendo as métricas do método 1 (WILCAR)
metrics_1 = {
    'mse_train': mse_train_1,
    'mse_test': mse_test_1,
    'r2_test': r2_test_1
}

In [ ]:
# Salvando as métricas do método 1 (WILCAR)
save_metrics(metrics_1, 'WILCAR')

## Método 1 (WILCAR) - Com Restrições nos Sinais dos Ganhos - MSE

### Inicialização dos Pesos
Obtenção dos pesos iniciais entre os neurônios de entrada e o primeiro neurônio oculto, já considerando restrições nos sinais dos ganhos.

In [ ]:
def objective_function(x, x_M, k, b, n_inputs):
    """
    Calcula a função objetivo baseada na comparação entre o modelo não-linear linearizado e o modelo obtido 
    pela Regressão Linear Múltipla para o ajuste dos pesos iniciais da rede neural.

    Essa função tem como objetivo minimizar a discrepância entre os pesos derivados da linearização do modelo
    não-linear via Série de Taylor e os coeficientes obtidos pela Regressão Linear Múltipla, promovendo uma
    inicialização eficaz dos pesos que conectam os neurônios de entrada ao primeiro neurônio oculto.

    Parâmetros:
        x (array): Um vetor de parâmetros do modelo, incluindo pesos e o bias.
        x_M (array): Médias das variáveis de entrada, usadas como ponto de equilíbrio para a linearização.
        k (array): Coeficientes obtidos pela Regressão Linear Múltipla.
        b (float): Termo de interceptação obtido pela Regressão Linear Múltipla.
        n_inputs (int): Número de variáveis de entrada.

    Retorna:
        float: O valor da função objetivo, representando a soma dos quadrados das diferenças entre
               a saída do modelo linearizado e os valores estimados pela regressão.
    """
    w, beta = x[:-1], x[-1]
    equations = [sigmoid(np.dot(w, x_M) + beta) - (sigmoid(np.dot(w, x_M) + beta) - (sigmoid(np.dot(w, x_M) + beta))**2) * np.dot(w, x_M) - b]
    for i in range(n_inputs):
        eq = (sigmoid(np.dot(w, x_M) + beta) - (sigmoid(np.dot(w, x_M) + beta))**2) * w[i] - k[i]
        equations.append(eq)
    return np.sum(np.square(equations))

In [ ]:
def gain_constraint(x, i, x_base, expected_signals, delta=0.1):
    """
    Define uma restrição para a otimização com base no sinal do ganho calculado de uma variável de entrada
    específica, garantindo que esteja alinhado com o sinal esperado.

    Esta função é utilizada como uma restrição no processo de otimização para assegurar que os sinais dos 
    ganhos das variáveis de entrada correspondam aos sinais esperados, conforme definido pelo usuário. 
    O ganho é calculado pela variação na saída do modelo em resposta a uma pequena alteração na entrada.

    Parâmetros:
        x (array): Um vetor de parâmetros do modelo, incluindo pesos e o bias.
        i (int): Índice da variável de entrada para a qual o ganho está sendo calculado.
        x_base (array): Vetor de entrada base para o cálculo do ganho.
        expected_signals (list): Lista dos sinais esperados para o ganho de cada variável de entrada.
        delta (float, opcional): Variação aplicada à variável de entrada para calcular o ganho. Padrão é 0.1.

    Retorna:
        float: Resultado da restrição. Um valor maior que 0 indica que o sinal do ganho está alinhado com o
               esperado, satisfazendo a restrição.
    """
    w, beta = x[:-1], x[-1]
    x_var = x_base.copy()
    x_var[i] += delta
    y_base = sigmoid(np.dot(w, x_base) + beta)
    new_y = sigmoid(np.dot(w, x_var) + beta)
    gain = (new_y - y_base) / delta
    if expected_signals[i] == 0:
        return 1
    return expected_signals[i] * gain - 0.01

In [ ]:
# Adiciona a coluna das constantes
inputs_all_cte = sm.add_constant(inputs_all, prepend=True)
res = sm.OLS(targets_all, inputs_all_cte).fit()  # Cria e ajusta o modelo
print(res.summary())  # Imprime os resultados

In [ ]:
# Definindo constantes
x_M = [np.mean(inputs_all[:, i]) for i in range(n_inputs)]
k = [res.params[i + 1] for i in range(n_inputs)]
b = res.params[0]

In [ ]:
# Chutes iniciais
x0 = np.random.rand(n_inputs + 1)

# Restrições
constraints = [{'type': 'ineq', 'fun': gain_constraint, 'args': (i, x_base, expected_signals)} for i in range(n_inputs)]

# Otimização
result_init = minimize(objective_function, x0, args=(x_M, k, b, n_inputs), constraints=constraints, method='SLSQP', options={'maxiter': 10000}, tol=1e-06)

if result_init.success:
    print("Otimização concluída com sucesso. Pesos ajustados:", result_init.x)
else:
    print("Otimização falhou:", result_init.message)

### Avaliação dos Sinais dos Ganhos do Modelo Inicial
O modelo inicial corresponde a um Perceptron.

In [ ]:
# Checagem dos valores dos ganhos associados aos pesos iniciais (utiliza-se como referência o primeiro registro do conjunto de testes)
x = test_inputs.T
x_base = np.array([x[k][0] for k in range(x.shape[0])]).reshape(x.shape[0], 1)

calculate_and_compare_gains(result_init.x, x_base, expected_signals, n_inputs)

### Integração dos Pesos Iniciais ao Modelo

In [ ]:
def extend_initial_params(x, n_neurons, input_size):
    """
    Estende os parâmetros iniciais para incluir pesos e bias adicionais para a camada de saída.

    Parâmetros:
        x (array): Parâmetros iniciais ajustados.
        n_neurons (int): Número de neurônios na camada oculta.
        input_size (int): Número de variáveis de entrada.

    Retorna:
        array: Parâmetros estendidos incluindo pesos e bias para a camada de saída.
    """
    W1_b1 = x  # Parâmetros atuais da otimização (pesos e bias para a primeira camada)
    
    # Inicializa parâmetros adicionais para W2 e b2 usando a inicialização Xavier
    fan_in = n_neurons  # Número de conexões de entrada para o neurônio na camada de saída
    fan_out = 1  # Como W2 conecta a camada oculta a um único neurônio de saída
    variance = 2 / (fan_in + fan_out)  # Variância de inicialização Xavier
    W2 = np.random.normal(0, np.sqrt(variance), (1, n_neurons))
    b2 = np.zeros(1)  # Normalmente os bias são inicializados com zero
    
    # Concatena todos os parâmetros em um vetor único
    extended_params = np.concatenate([W1_b1, W2.flatten(), b2])
    return extended_params

In [ ]:
def reshape_initial_weights(x, n_neurons, input_size):
    """
    Redimensiona os pesos iniciais estendidos para formar as matrizes e vetores necessários para a rede neural.

    Parâmetros:
        x (array): Parâmetros iniciais estendidos.
        n_neurons (int): Número de neurônios na camada oculta.
        input_size (int): Número de variáveis de entrada.

    Retorna:
        dict: Dicionário contendo as matrizes e vetores de pesos e bias para cada camada.
    """
    n_W1 = n_neurons * input_size
    n_b1 = n_neurons
    n_W2 = n_neurons  # Como W2 conecta cada neurônio na camada oculta à saída
    n_b2 = 1  # Bias único para o neurônio de saída
    
    assert len(x) == n_W1 + n_b1 + n_W2 + n_b2, "Comprimento do vetor de parâmetros inicial incorreto"
    
    W1 = x[:n_W1].reshape(n_neurons, input_size)
    b1 = x[n_W1:n_W1 + n_b1].reshape(n_neurons, 1)
    W2 = x[n_W1 + n_b1:n_W1 + n_b1 + n_W2].reshape(1, n_neurons)
    b2 = x[-n_b2:].reshape(1, 1)
    
    return {'W1': W1, 'b1': b1, 'W2': W2, 'b2': b2}

In [ ]:
input_size = train_inputs.shape[1]
n_neurons = 1  # Um neurônio na camada oculta

# Estende os parâmetros iniciais se a otimização foi bem-sucedida
if result_init.success:
    extended_params = extend_initial_params(result_init.x, n_neurons, input_size)
    initial_params = reshape_initial_weights(extended_params, n_neurons, input_size)

### Definição / Implementação do Método 1 com Restrição de Sinais de Ganhos - MSE

#### Funções Auxiliares

In [ ]:
def objective(params, x, y, n, input_size):
    """
    Calcula a função objetivo (erro quadrático médio) para a rede neural.

    Parâmetros:
        params (dict): Parâmetros da rede neural.
        x (array): Dados de entrada.
        y (array): Dados de saída esperados.
        n (int): Número de neurônios na camada oculta.
        input_size (int): Número de variáveis de entrada.

    Retorna:
        float: Erro quadrático médio (MSE) entre a saída predita e a saída esperada.
    """
    W1 = params['W1']
    b1 = params['b1']
    W2 = params['W2']
    b2 = params['b2']
    
    Z1 = np.dot(W1, x) + b1
    A1 = sigmoid(Z1)
    Z2 = np.dot(W2, A1) + b2
    A2 = sigmoid(Z2)
    
    return mse(y, A2.flatten())

def sigmoid(x):
    """Função sigmoide numericamente estável."""
    clipped_x = np.clip(x, -500, 500)  # Limita os valores para evitar extremos em exp
    return np.where(clipped_x >= 0, 
                    1 / (1 + np.exp(-clipped_x)), 
                    np.exp(clipped_x) / (1 + np.exp(clipped_x)))

def initialize_weights(n, input_size, old_weights=None, x_train=None, expected_signals=None):
    """
    Inicializa os pesos usando a inicialização Xavier, com opção de incluir pesos antigos e ajustes para satisfazer as restrições de ganho.

    Parâmetros:
        n (int): Número de neurônios na camada oculta.
        input_size (int): Número de variáveis de entrada.
        old_weights (dict ou np.ndarray, opcional): Pesos antigos para reutilização.
        x_train (array, opcional): Dados de entrada para verificar os ganhos.
        expected_signals (list, opcional): Sinais esperados dos ganhos.

    Retorna:
        dict: Dicionário com os pesos e biases inicializados.
    """
    while True:
        W1_new = np.random.uniform(-math.sqrt(6 / (input_size + n)), math.sqrt(6 / (input_size + n)), (n, input_size))
        b1_new = np.zeros((n, 1))
        W2_new = np.random.uniform(-math.sqrt(6 / (n + 1)), math.sqrt(6 / (n + 1)), (1, n))
        b2_new = np.zeros((1, 1))

        if old_weights is not None:
            if isinstance(old_weights, dict):
                num_old_neurons = min(n, old_weights['W1'].shape[0])
                W1_new[:num_old_neurons, :] = old_weights['W1'][:num_old_neurons, :]
                b1_new[:num_old_neurons, :] = old_weights['b1'][:num_old_neurons, :]
                W2_new[:, :num_old_neurons] = old_weights['W2'][:, :num_old_neurons]
                b2_new[:] = old_weights['b2']
            elif isinstance(old_weights, np.ndarray):
                num_params_per_neuron = input_size + 1
                W1_new[0, :] = old_weights[:input_size]
                b1_new[0, 0] = old_weights[input_size]

        params = {'W1': W1_new, 'b1': b1_new, 'W2': W2_new, 'b2': b2_new}

        if x_train is not None and expected_signals is not None:
            gains_satisfy = True
            for i in range(input_size):
                if expected_signals[i] != 0:
                    gain = calculate_gain(params, x_train, n, input_size, i)
                    if np.sign(gain) != expected_signals[i]:
                        gains_satisfy = False
                        break
            if gains_satisfy:
                return params
        else:
            return params

def predict_network(params, x):
    """
    Faz previsões usando a rede neural com os parâmetros fornecidos.

    Parâmetros:
        params (dict): Parâmetros da rede neural.
        x (array): Dados de entrada.

    Retorna:
        array: Previsões da rede neural.
    """
    W1 = params['W1']
    b1 = params['b1']
    W2 = params['W2']
    b2 = params['b2']

    Z1 = np.dot(W1, x) + b1
    A1 = sigmoid(Z1)
    Z2 = np.dot(W2, A1) + b2
    A2 = sigmoid(Z2)

    return A2

def calculate_gain(params, x, n, input_size, var_index):
    """
    Calcula o ganho de uma variável de entrada específica.

    Parâmetros:
        params (dict): Parâmetros da rede neural.
        x (array): Dados de entrada.
        n (int): Número de neurônios na camada oculta.
        input_size (int): Número de variáveis de entrada.
        var_index (int): Índice da variável de entrada.

    Retorna:
        float: Ganho calculado.
    """
    delta = 0.1
    x_perturbed = np.copy(x)
    x_perturbed[var_index, :] += delta

    y_base = predict_network(params, x)
    y_perturbed = predict_network(params, x_perturbed)
    
    gain = np.mean((y_perturbed - y_base) / delta, axis=1)
    return gain[0]

def gain_constraint(params, x, n, input_size, var_index, expected_gain):
    """
    Define uma restrição de ganho para a otimização.

    Parâmetros:
        params (dict): Parâmetros da rede neural.
        x (array): Dados de entrada.
        n (int): Número de neurônios na camada oculta.
        input_size (int): Número de variáveis de entrada.
        var_index (int): Índice da variável de entrada.
        expected_gain (int): Sinal do ganho esperado.

    Retorna:
        float: Resultado da restrição.
    """
    gain = calculate_gain(params, x, n, input_size, var_index)
    margin = 0.01
    if expected_gain == 1:
        return gain - margin
    elif expected_gain == -1:
        return -gain - margin
    return 0

def calculate_and_compare_gains(best_params, x_base, expected_signals, input_size):
    """
    Calcula os ganhos e os compara com os sinais esperados.

    Parâmetros:
        best_params (dict): Melhores parâmetros da rede neural.
        x_base (array): Dados de entrada base.
        expected_signals (list): Sinais esperados dos ganhos.
        input_size (int): Número de variáveis de entrada.

    Retorna:
        None
    """
    delta = 0.1  # Magnitude da perturbação
    gains_calculated = []
    
    for i in range(input_size):
        x_perturbed = np.copy(x_base)
        x_perturbed[i, :] += delta
        y_base = predict_network(best_params, x_base)
        y_perturbed = predict_network(best_params, x_perturbed)
        gain_i = (y_perturbed - y_base) / delta
        gains_calculated.append(np.mean(gain_i))

    for i, (gain_calculated, signal_expected) in enumerate(zip(gains_calculated, expected_signals)):
        signal_calculated = np.sign(gain_calculated)
        if signal_expected == 0:
            print(f"Variável {i+1}: Ganho calculado = {signal_calculated}, Ganho esperado = Desconhecido")
        elif signal_calculated == signal_expected:
            print(f"Variável {i+1}: Ganho calculado = {signal_calculated}, Ganho esperado = {signal_expected} [Conforme]")
        else:
            print(f"Variável {i+1}: Ganho calculado = {signal_calculated}, Ganho esperado = {signal_expected} [Divergente]")

#### Conversão de Parâmetros

In [ ]:
def param_dict_to_vector(params):
    """Converte um dicionário de parâmetros em um vetor plano."""
    W1_flat = params['W1'].flatten()
    b1_flat = params['b1'].flatten()
    W2_flat = params['W2'].flatten()
    b2_flat = params['b2'].flatten()
    return np.concatenate([W1_flat, b1_flat, W2_flat, b2_flat])

def param_vector_to_dict(vector, n, input_size):
    """Converte um vetor plano de volta para um dicionário de parâmetros com os formatos corretos."""
    n_W1 = n * input_size
    n_b1 = n
    n_W2 = n
    n_b2 = 1

    W1 = vector[:n_W1].reshape(n, input_size)
    b1 = vector[n_W1:n_W1 + n_b1].reshape(n, 1)
    W2 = vector[n_W1 + n_b1:n_W1 + n_b1 + n_W2].reshape(1, n)
    b2 = vector[-n_b2:].reshape(1, 1)

    return {'W1': W1, 'b1': b1, 'W2': W2, 'b2': b2}

#### Treinamento da Rede Neural

In [ ]:
def train_network(train_inputs, train_targets, test_inputs, expected_signals, result_init):
    input_size = train_inputs.shape[1]
    x_train = train_inputs.T
    y_train = train_targets.T
    x_test = test_inputs.T
    y_test = test_targets.T
    mse_train = []
    mse_test = []
    r2_train = []
    r2_test = []
    rmse_train = []
    rmse_test = []
    best_metrics = {'mse': float('inf'), 'r2': -float('inf'), 'rmse': float('inf')}
    best_num_neurons = 0
    last_params = None

    gain_matches = []

    for n in range(1, 251):
        print(f"Treinando com {n} neurônios...")
        if n == 1:
            params = initialize_weights(n, input_size, result_init.x, x_train, expected_signals)
        else:
            params = initialize_weights(n, input_size, last_params, x_train, expected_signals)

        W0 = param_dict_to_vector(params)

        constraints = [{'type': 'ineq', 'fun': lambda W, x, n, input_size, i, sig: gain_constraint(param_vector_to_dict(W, n, input_size), x, n, input_size, i, sig),
                        'args': (x_train, n, input_size, i, expected_signals[i])}
                       for i in range(n_inputs) if expected_signals[i] != 0]

        result = minimize(lambda W: objective(param_vector_to_dict(W, n, input_size), x_train, y_train, n, input_size),
                          W0,
                          method='SLSQP',
                          constraints=constraints,
                          options={'disp': True, 'maxiter': 1000, 'ftol': 1e-05})

        if result.success:
            optimized_params = param_vector_to_dict(result.x, n, input_size)
            current_mse = objective(optimized_params, x_train, y_train, n, input_size)
            current_rmse = np.sqrt(current_mse)
            mse_train.append(current_mse)
            rmse_train.append(current_rmse)
            last_params = optimized_params

            y_test_pred = predict_network(optimized_params, x_test)
            test_mse = mse(y_test, y_test_pred.flatten())
            test_rmse = np.sqrt(test_mse)
            mse_test.append(test_mse)
            rmse_test.append(test_rmse)

            r2_test_value = r2_score(y_test, y_test_pred.flatten())
            r2_test.append(r2_test_value)

            all_gains_match = True
            for i, expected_gain in enumerate(expected_signals):
                calculated_gain = calculate_gain(optimized_params, x_test, n, input_size, i)
                gain_match = np.sign(calculated_gain) == expected_gain
                all_gains_match &= gain_match

                if not gain_match:
                    print(f"Treinamento interrompido em {n} neurônios. Problema de convergência na variável {i+1}. Ganho calculado: {np.sign(calculated_gain)}, Ganho esperado: {expected_gain}")
                    return mse_train, mse_test, r2_train, r2_test, rmse_train, rmse_test, best_metrics, best_params, gain_matches

            gain_matches.append(all_gains_match)

            if r2_test_value > best_metrics['r2']:
                best_metrics['r2'] = r2_test_value
                best_metrics['mse'] = test_mse
                best_metrics['rmse'] = test_rmse
                best_params = optimized_params

            print(f"Treinamento com {n} neurônios concluído, R2 de Teste: {r2_test[-1]}, RMSE de Teste: {test_rmse}, Correspondência de Ganhos: {all_gains_match}")
        else:
            print(f"Treinamento interrompido em {n} neurônios. Otimização não convergiu. Motivo: {result.message}")
            return mse_train, mse_test, r2_train, r2_test, rmse_train, rmse_test, best_metrics, best_params, gain_matches

    print('ANÁLISE DE GANHOS - MELHOR MODELO - WILCAR C/ RESTRIÇÕES')
    calculate_and_compare_gains(best_params, x_test, expected_signals, input_size)

    return mse_train, mse_test, r2_train, r2_test, rmse_train, rmse_test, best_metrics, best_params, gain_matches

ini = time.time()
# Assume train_inputs, train_targets, test_inputs, expected_signals, result_init são definidos
mse_train, mse_test, r2_train, r2_test, rmse_train, rmse_test, best_metrics, best_params, gain_matches = train_network(train_inputs, train_targets, test_inputs, expected_signals, result_init)
print("Treinamento concluído. Melhor MSE de Teste:", best_metrics['mse'])
print("Treinamento concluído. Melhor RMSE de Teste:", best_metrics['rmse'])

### Análise da Performance do Método 1 com Restrições dos Sinais dos Ganhos - MSE

In [ ]:
# Finalizando e exibindo contagem de tempo de processamento:
fim = time.time()
print("Tempo de processamento (s):", fim-ini)

# Guardando Métricas:
mse_train1_cons_mse = mse_train
mse_test1_cons_mse = mse_test
r2_train1_cons_mse = r2_train
r2_test1_cons_mse = r2_test
rmse_train1_cons_mse = rmse_train
rmse_test1_cons_mse = rmse_test
best_model1_cons_params_mse = best_params

In [ ]:
print("Tempo de processamento (s):", fim-ini)

# Exibindo, em tela, R2 máximo de teste:
print("R2 máximo de teste obtido:", max(r2_test1_cons_mse))

# Exibindo, em tela, número ótimo de neurônios ocultos:
print("Número ótimo de neurônios ocultos:", (r2_test1_cons_mse.index(max(r2_test1_cons_mse))) + 1)

# Análise final de desempenho do Método 1 com restrições de ganhos
print("MSE de teste final com o melhor modelo:", best_metrics['mse'])
print("RMSE de teste final com o melhor modelo:", best_metrics['rmse'])

In [ ]:
# Criação e exibição de gráfico do MSE de treino:
plt.plot(mse_train1_cons_mse)
plt.ylabel('MSE de Treinamento')
plt.xlabel('Neurônios Ocultos')
plt.legend(['Constrained WILCAR (MSE as obj. fnc.)'])
plt.show()

In [ ]:
# Criação e exibição de gráfico do RMSE de treino:
plt.plot(rmse_train1_cons_mse)
plt.ylabel('RMSE de Treinamento')
plt.xlabel('Neurônios Ocultos')
plt.legend(['Constrained WILCAR (MSE as obj. fnc.)'])
plt.show()

In [ ]:
# Criação e exibição de gráfico do R2 de treino:
#plt.plot(r2_train1_cons)
#plt.ylabel('R2 de Treinamento')
#plt.xlabel('Neurônios Ocultos')
#plt.legend(['Constrained WILCAR'])
#plt.show()

In [ ]:
# Criação e exibição de gráfico do MSE de teste:
plt.plot(mse_test1_cons_mse)
plt.ylabel('MSE de Teste')
plt.xlabel('Neurônios Ocultos')
plt.legend(['Constrained WILCAR (MSE as obj. fnc.)'])
plt.show()

In [ ]:
# Criação e exibição de gráfico do RMSE de teste:
plt.plot(rmse_test1_cons_mse)
plt.ylabel('RMSE de Teste')
plt.xlabel('Neurônios Ocultos')
plt.legend(['Constrained WILCAR (MSE as obj. fnc.)'])
plt.show()

In [ ]:
# Criação e exibição de gráfico do R2 de teste:
plt.plot(r2_test1_cons_mse)
plt.ylabel('R2 de Teste')
plt.xlabel('Neurônios Ocultos')
plt.legend(['Constrained WILCAR (MSE as obj. fnc.)'])
plt.show()

### Salvando Métricas do Método 1 com Restrições dos Sinais dos Ganhos (Constrained WILCAR) - MSE

In [ ]:
metrics_1_cons_mse = {
    'mse_train': mse_train1_cons_mse,
    'mse_test': mse_test1_cons_mse,
    'r2_test': r2_test1_cons_mse,
    'rmse_train': rmse_train1_cons_mse,
    'rmse_test': rmse_test1_cons_mse
}

In [ ]:
save_metrics(metrics_1_cons_mse, 'Constrained_WILCAR_MSE')

## Método 1 (WILCAR) - Com Restrições nos Sinais dos Ganhos - RMSE

### Definição / Implementação do Método 1 com Restrição de Sinais de Ganhos - RMSE

#### Funções Auxiliares

In [ ]:
def objective_rmse(params, x, y, n, input_size):
    """
    Calcula a função objetivo (erro quadrático médio) para a rede neural.

    Parâmetros:
        params (dict): Parâmetros da rede neural.
        x (array): Dados de entrada.
        y (array): Dados de saída esperados.
        n (int): Número de neurônios na camada oculta.
        input_size (int): Número de variáveis de entrada.

    Retorna:
        float: Erro quadrático médio (RMSE) entre a saída predita e a saída esperada.
    """
    W1 = params['W1']
    b1 = params['b1']
    W2 = params['W2']
    b2 = params['b2']
    
    Z1 = np.dot(W1, x) + b1
    A1 = sigmoid(Z1)
    Z2 = np.dot(W2, A1) + b2
    A2 = sigmoid(Z2)
    
    return np.sqrt(mse(y, A2.flatten()))

#### Treinamento da Rede Neural

In [ ]:
def train_network_rmse(train_inputs, train_targets, test_inputs, expected_signals, result_init):
    input_size = train_inputs.shape[1]
    x_train = train_inputs.T
    y_train = train_targets.T
    x_test = test_inputs.T
    y_test = test_targets.T
    mse_train = []
    mse_test = []
    r2_train = []
    r2_test = []
    rmse_train = []
    rmse_test = []
    best_metrics = {'mse': float('inf'), 'r2': -float('inf'), 'rmse': float('inf')}
    best_num_neurons = 0
    last_params = None

    gain_matches = []

    for n in range(1, 251):
        print(f"Treinando com {n} neurônios...")
        if n == 1:
            params = initialize_weights(n, input_size, result_init.x, x_train, expected_signals)
        else:
            params = initialize_weights(n, input_size, last_params, x_train, expected_signals)

        W0 = param_dict_to_vector(params)

        constraints = [{'type': 'ineq', 'fun': lambda W, x, n, input_size, i, sig: gain_constraint(param_vector_to_dict(W, n, input_size), x, n, input_size, i, sig),
                        'args': (x_train, n, input_size, i, expected_signals[i])}
                       for i in range(n_inputs) if expected_signals[i] != 0]

        result = minimize(lambda W: objective_rmse(param_vector_to_dict(W, n, input_size), x_train, y_train, n, input_size),
                          W0,
                          method='SLSQP',
                          constraints=constraints,
                          options={'disp': True, 'maxiter': 1000, 'ftol': 1e-05})

        if result.success:
            optimized_params = param_vector_to_dict(result.x, n, input_size)
            current_rmse = objective_rmse(optimized_params, x_train, y_train, n, input_size)
            rmse_train.append(current_rmse)
            current_mse = mse(y_train, predict_network(optimized_params, x_train).flatten())
            mse_train.append(current_mse)
            last_params = optimized_params

            y_test_pred = predict_network(optimized_params, x_test)
            test_mse = mse(y_test, y_test_pred.flatten())
            mse_test.append(test_mse)
            test_rmse = np.sqrt(test_mse)
            rmse_test.append(test_rmse)

            r2_test_value = r2_score(y_test, y_test_pred.flatten())
            r2_test.append(r2_test_value)

            all_gains_match = True
            for i, expected_gain in enumerate(expected_signals):
                calculated_gain = calculate_gain(optimized_params, x_test, n, input_size, i)
                gain_match = np.sign(calculated_gain) == expected_gain
                all_gains_match &= gain_match

                if not gain_match:
                    print(f"Treinamento interrompido em {n} neurônios. Problema de convergência na variável {i+1}. Ganho calculado: {np.sign(calculated_gain)}, Ganho esperado: {expected_gain}")
                    return mse_train, mse_test, r2_train, r2_test, rmse_train, rmse_test, best_metrics, best_params, gain_matches

            gain_matches.append(all_gains_match)

            if r2_test_value > best_metrics['r2']:
                best_metrics['r2'] = r2_test_value
                best_metrics['mse'] = test_mse
                best_metrics['rmse'] = test_rmse
                best_params = optimized_params

            print(f"Treinamento com {n} neurônios concluído, R2 de Teste: {r2_test[-1]}, Correspondência de Ganhos: {all_gains_match}")
        else:
            print(f"Treinamento interrompido em {n} neurônios. Otimização não convergiu. Motivo: {result.message}")
            return mse_train, mse_test, r2_train, r2_test, rmse_train, rmse_test, best_metrics, best_params, gain_matches

    print('ANÁLISE DE GANHOS - MELHOR MODELO - WILCAR C/ RESTRIÇÕES')
    calculate_and_compare_gains(best_params, x_test, expected_signals, input_size)

    return mse_train, mse_test, r2_train, r2_test, rmse_train, rmse_test, best_metrics, best_params, gain_matches

ini = time.time()
# Assume train_inputs, train_targets, test_inputs, expected_signals, result_init são definidos
mse_train, mse_test, r2_train, r2_test, rmse_train, rmse_test, best_metrics, best_params, gain_matches = train_network_rmse(train_inputs, train_targets, test_inputs, expected_signals, result_init)
print("Treinamento concluído. Melhor MSE de Teste:", best_metrics['mse'])
print("Treinamento concluído. Melhor RMSE de Teste:", best_metrics['rmse'])


### Análise da Performance do Método 1 com Restrições dos Sinais dos Ganhos - RMSE

In [ ]:
# Finalizando e exibindo contagem de tempo de processamento:
fim = time.time()
print("Tempo de processamento (s):", fim-ini)

# Guardando Métricas:
mse_train1_cons_rmse = mse_train
mse_test1_cons_rmse = mse_test
r2_train1_cons_rmse = r2_train
r2_test1_cons_rmse = r2_test
rmse_train1_cons_rmse = rmse_train
rmse_test1_cons_rmse = rmse_test
best_model1_cons_params_rmse = best_params

In [ ]:
# Exibindo, em tela, R2 máximo de teste:
print("R2 máximo de teste obtido:", max(r2_test1_cons_rmse))

# Exibindo, em tela, número ótimo de neurônios ocultos:
print("Número ótimo de neurônios ocultos:", (r2_test1_cons_rmse.index(max(r2_test1_cons_rmse))) + 1)

# Análise final de desempenho do Método 1 com restrições de ganhos
print("MSE de teste final com o melhor modelo:", best_metrics['mse'])
print("RMSE de teste final com o melhor modelo:", best_metrics['rmse'])

In [ ]:
# Criação e exibição de gráfico do MSE de treino:
plt.plot(mse_train1_cons_rmse)
plt.ylabel('MSE de Treinamento')
plt.xlabel('Neurônios Ocultos')
plt.legend(['Constrained WILCAR (RMSE as obj. fnc.)'])
plt.show()

In [ ]:
# Criação e exibição de gráfico do RMSE de treino:
plt.plot(rmse_train1_cons_rmse)
plt.ylabel('RMSE de Treinamento')
plt.xlabel('Neurônios Ocultos')
plt.legend(['Constrained WILCAR (RMSE as obj. fnc.)'])
plt.show()

In [ ]:
# Criação e exibição de gráfico do MSE de teste:
plt.plot(mse_test1_cons_rmse)
plt.ylabel('MSE de Teste')
plt.xlabel('Neurônios Ocultos')
plt.legend(['Constrained WILCAR (RMSE as obj. fnc.)'])
plt.show()

In [ ]:
# Criação e exibição de gráfico do RMSE de teste:
plt.plot(rmse_test1_cons_rmse)
plt.ylabel('RMSE de Teste')
plt.xlabel('Neurônios Ocultos')
plt.legend(['Constrained WILCAR (RMSE as obj. fnc.)'])
plt.show()

In [ ]:
# Criação e exibição de gráfico do R2 de teste:
plt.plot(r2_test1_cons_rmse)
plt.ylabel('R2 de Teste')
plt.xlabel('Neurônios Ocultos')
plt.legend(['Constrained WILCAR (RMSE as obj. fnc.)'])
plt.show()

### Salvando Métricas do Método 1 com Restrições dos Sinais dos Ganhos (Constrained WILCAR - RMSE)

In [ ]:
metrics_1_cons_rmse = {
    'mse_train': mse_train1_cons_rmse,
    'mse_test': mse_test1_cons_rmse,
    'r2_test': r2_test1_cons_rmse,
    'rmse_train': rmse_train1_cons_rmse,
    'rmse_test': rmse_test1_cons_rmse
}
save_metrics(metrics_1_cons_rmse, 'Constrained_WILCAR_RMSE')

## Método 2 (RIXM)
Modelo SFNN considerando inicialização de pesos a partir de uma inicialização randômica (Método de Xavier) e bias inicializados com valor nulo.

### Implementação do Método 2
Aproveita-se a classe definida para o Método 1.

In [ ]:
# Inicializando variáveis para guardar métricas
ini = time.time()
mse_train_2 = []
r2_train_2 = []
mse_test_2 = []
r2_test_2 = []

# Inicialização de variáveis para rastrear o melhor modelo
best_r2_test_2 = -np.inf  # Inicializa com infinito negativo para garantir que qualquer modelo seja melhor que este inicialmente
best_model_2 = None  # Para guardar o melhor modelo
best_n_2 = 0  # Para guardar o número de neurônios do melhor modelo

# Loop para treinar o modelo com diferentes números de neurônios ocultos
for n in range(1, 251):
    if n == 1:
        model = Method1(train_inputs.T, train_targets.T, n, initial_weights=None)
    else:
        model = Method1(train_inputs.T, train_targets.T, n, previous_model=None)
    
    mse_train, r2_train, mse_test, r2_test = model.train_test()
    
    mse_train_2.append(mse_train)
    r2_train_2.append(r2_train)
    mse_test_2.append(mse_test)
    r2_test_2.append(r2_test)
    
    print(f"Neurônios Ocultos: {n} - MSE de Treinamento = {mse_train}, R2 de Treinamento = {r2_train}, MSE de Teste = {mse_test}, R2 de Teste = {r2_test}")
    
    # Atualiza o melhor modelo se o atual for melhor que o registrado anteriormente
    if r2_test > best_r2_test_2:
        best_r2_test_2 = r2_test
        best_model_2 = model  # Salva o modelo atual como o melhor modelo
        best_n_2 = n  # Salva o número de neurônios do melhor modelo

    # Verifica os ganhos para cada modelo treinado
    x_base = test_inputs.T
    delta = 0.1
    ganhos_calculados = []
    for i in range(n_inputs):
        x_perturbed = np.copy(x_base)
        x_perturbed[i, :] += delta  # Perturba cada variável de entrada individualmente
        y_base = model.predict(x_base)
        y_perturbed = model.predict(x_perturbed)
        ganho_i = (y_perturbed - y_base) / delta
        ganhos_calculados.append(np.mean(ganho_i)) # Tira a média de todos os ganhos calculados, por variável

    # Compara os ganhos calculados com os sinais de ganhos esperados
    for i, (ganho_calculado, sinal_esperado) in enumerate(zip(ganhos_calculados, expected_signals)):
        sinal_calculado = np.sign(ganho_calculado)
        
        if sinal_esperado == 0:
            print(f"Modelo {n}, Variável {i+1}: Ganho calculado = {sinal_calculado}, Ganho esperado = Desconhecido")
        elif sinal_calculado == sinal_esperado:
            print(f"Modelo {n}, Variável {i+1}: Ganho calculado = {sinal_calculado}, Ganho esperado = {sinal_esperado} [Conforme]")
        else:
            print(f"Modelo {n}, Variável {i+1}: Ganho calculado = {sinal_calculado}, Ganho esperado = {sinal_esperado} [Divergente]")
    
    previous_model = model

end = time.time()
print(f"Método concluído em {end - ini} segundos.")
print(f"Melhor Modelo: {best_n_2} Neurônios, R2 de Teste = {best_r2_test_2}")

### Análise da Performance do Método 2

In [ ]:
# Define o conjunto de teste como referência para o cálculo dos ganhos
x_base = test_inputs.T

# Definindo delta (variação nas entradas)
delta = 0.1

# Calcula os ganhos para cada variável de entrada
ganhos_calculados = []
for i in range(n_inputs):
    x_perturbed = np.copy(x_base)
    x_perturbed[i, :] += delta  # Perturba cada variável de entrada individualmente
    y_base = best_model_2.predict(x_base)
    y_perturbed = best_model_2.predict(x_perturbed)
    ganho_i = (y_perturbed - y_base) / delta
    ganhos_calculados.append(np.mean(ganho_i)) # Tira a média de todos os ganhos calculados, por variável

print('ANÁLISE DE GANHOS - MELHOR MODELO - RIXM')

# Compara os ganhos calculados com os sinais de ganhos esperados
for i, (ganho_calculado, sinal_esperado) in enumerate(zip(ganhos_calculados, expected_signals)):
    sinal_calculado = np.sign(ganho_calculado)
    
    if sinal_esperado == 0:
        print(f"Variável {i+1}: Ganho calculado = {sinal_calculado}, Ganho esperado = Desconhecido")
    elif sinal_calculado == sinal_esperado:
        print(f"Variável {i+1}: Ganho calculado = {sinal_calculado}, Ganho esperado = {sinal_esperado} [Conforme]")
    else:
        print(f"Variável {i+1}: Ganho calculado = {sinal_calculado}, Ganho esperado = {sinal_esperado} [Divergente]")

In [ ]:
# Imprime o número ótimo de neurônios ocultos e o melhor R2 de teste
print("Número ótimo de neurônios ocultos:", best_n_2)
print("Melhor R2 de Teste:", best_r2_test_2)

In [ ]:
# Criação e exibição de Gráfico do MSE de treino:
plt.plot(mse_train_2)
plt.ylabel('MSE de Treinamento')
plt.xlabel('Neurônios Ocultos')
plt.legend(['RIXM'])
plt.show()

In [ ]:
# Criação e exibição de Gráfico do MSE de teste:
plt.plot(mse_test_2)
plt.ylabel('MSE de Teste')
plt.xlabel('Neurônios Ocultos')
plt.legend(['RIXM'])
plt.show()

In [ ]:
# Criação e exibição de Gráfico do R2 de treino:
#plt.plot(r2_train_2)
#plt.ylabel('Train R2')
#plt.xlabel('Hidden Nodes')
#plt.legend(['RIXM'])
#plt.show()

In [ ]:
# Criação e exibição de Gráfico do R2 de teste:
plt.plot(r2_test_2)
plt.ylabel('R2 de Teste')
plt.xlabel('Neurônios Ocultos')
plt.legend(['RIXM'])
plt.show()

### Salvando Métricas do Método 2 (RIXM)

In [ ]:
metrics_2 = {
    'mse_train': mse_train_2,
    'mse_test': mse_test_2,
    'r2_train': r2_train_2,
    'r2_test': r2_test_2
}

In [ ]:
save_metrics(metrics_2, 'RIXM')

## Método 2 (RIXM) - Com Restrições dos Sinais dos Ganhos

### Implementação do Método 2 (RIXM) - Com Restrições dos Sinais dos Ganhos

In [ ]:
class Method2_CONS:
    def __init__(self, input_size, output_size=1):
        self.input_size = input_size
        self.output_size = output_size

    @staticmethod
    def sigmoid(x):
        clipped_x = np.clip(x, -500, 500)  # Limita valores para evitar extremos em exp
        return np.where(clipped_x >= 0, 
                        1 / (1 + np.exp(-clipped_x)), 
                        np.exp(clipped_x) / (1 + np.exp(clipped_x)))

    def initialize_weights(self, n, x_train, expected_signals):
        """
        Inicializa os pesos usando a inicialização Xavier e ajusta até que os ganhos satisfaçam as restrições.
        """
        init_range = np.sqrt(6 / (self.input_size + n))
        while True:
            W1 = np.random.uniform(-init_range, init_range, (n, self.input_size))
            b1 = np.zeros((n, 1))
            W2 = np.random.uniform(-init_range, init_range, (1, n))
            b2 = np.zeros((1, 1))
            params = {'W1': W1, 'b1': b1, 'W2': W2, 'b2': b2}

            # Verifica se os ganhos iniciais satisfazem as restrições
            all_gains_match = True
            for i, expected_gain in enumerate(expected_signals):
                if expected_gain != 0:
                    calculated_gain = self.calculate_gain(params, x_train, i)
                    gain_match = np.sign(calculated_gain) == expected_gain
                    all_gains_match &= gain_match
                    if not gain_match:
                        break

            if all_gains_match:
                break

        return params

    def predict(self, x, params):
        Z1 = np.dot(params['W1'], x) + params['b1']
        A1 = self.sigmoid(Z1)
        Z2 = np.dot(params['W2'], A1) + params['b2']
        A2 = self.sigmoid(Z2)
        return A2

    def objective(self, params, x, y):
        A2 = self.predict(x, params)
        return mse(y, A2.flatten())

    def calculate_gain(self, params, x, var_index, delta=0.1):
        x_perturbed = np.copy(x)
        x_perturbed[var_index, :] += delta
        y_base = self.predict(x, params)
        y_perturbed = self.predict(x_perturbed, params)
        gain = np.mean((y_perturbed - y_base) / delta, axis=1)
        return gain[0]

    def gain_constraint(self, params, x, var_index, expected_gain):
        actual_gain = self.calculate_gain(params, x, var_index)
        if expected_gain == 1:
            return actual_gain - 0.1
        elif expected_gain == -1:
            return -actual_gain - 0.1
        return 0

    def train_network(self, train_inputs, train_targets, test_inputs, test_targets, expected_signals):
        x_train = train_inputs.T
        y_train = train_targets.T
        x_test = test_inputs.T
        y_test = test_targets.T

        mse_train = []
        mse_test = []
        r2_train = []
        r2_test = []
        gain_matches = []
        best_metrics = {'mse': float('inf'), 'r2': -float('inf')}
        best_params = None

        for n in range(1, 251):
            print(f"Treinando com {n} neurônios...")
            params = self.initialize_weights(n, x_train, expected_signals)
            W0 = np.concatenate([params['W1'].flatten(), params['b1'].flatten(), params['W2'].flatten(), params['b2'].flatten()])

            constraints = [{'type': 'ineq', 'fun': lambda W, x=x_train, n=n, i=i, sig=expected_signals[i]: self.gain_constraint(self.param_vector_to_dict(W, n), x, i, sig),
                            'args': []} for i in range(self.input_size) if expected_signals[i] != 0]

            result = minimize(lambda W: self.objective(self.param_vector_to_dict(W, n), x_train, y_train),
                              W0, method='SLSQP', constraints=constraints,
                              options={'disp': True, 'maxiter': 1000, 'ftol': 1e-05})

            if result.success:
                optimized_params = self.param_vector_to_dict(result.x, n)
                current_mse = self.objective(optimized_params, x_train, y_train)
                mse_train.append(current_mse)

                y_test_pred = self.predict(x_test, optimized_params)
                test_mse = mse(y_test, y_test_pred.flatten())
                mse_test.append(test_mse)
                current_r2 = r2_score(y_test, y_test_pred.flatten())
                r2_test.append(current_r2)

                all_gains_match = True
                for i, expected_gain in enumerate(expected_signals):
                    calculated_gain = self.calculate_gain(optimized_params, x_test, i)
                    gain_match = np.sign(calculated_gain) == expected_gain
                    all_gains_match &= gain_match
                
                gain_matches.append(all_gains_match)

                if current_r2 > best_metrics['r2']:
                    best_metrics['r2'] = current_r2
                    best_metrics['mse'] = test_mse
                    best_params = optimized_params

                print(f"Treinamento com {n} neurônios concluído, R2 de Teste: {current_r2}, Correspondência de Ganhos: {all_gains_match}")
            else:
                print(f"Treinamento com {n} neurônios falhou na otimização.")
                break

            if not all_gains_match:
                print(f"Treinamento interrompido em {n} neurônios. Problema de correspondência de ganhos.")
                break

        # Análise final dos ganhos para o melhor modelo
        final_gain_matches = []
        if best_params:
            for i, expected_gain in enumerate(expected_signals):
                final_calculated_gain = self.calculate_gain(best_params, x_test, i)
                final_gain_matches.append(np.sign(final_calculated_gain) == expected_gain)

        return mse_train, mse_test, r2_train, r2_test, best_metrics, best_params, gain_matches, final_gain_matches

    def param_vector_to_dict(self, vector, n):
        n_W1 = n * self.input_size
        n_b1 = n
        n_W2 = n
        n_b2 = 1  # Porque W2 tem apenas uma linha e b2 é um único termo de bias para a saída

        W1 = vector[:n_W1].reshape(n, self.input_size)
        b1 = vector[n_W1:n_W1 + n_b1].reshape(n, 1)
        W2 = vector[n_W1 + n_b1:n_W1 + n_b1 + n_W2].reshape(1, n)
        b2 = vector[-n_b2:].reshape(1, 1)

        return {'W1': W1, 'b1': b1, 'W2': W2, 'b2': b2}

ini = time.time()
nn = Method2_CONS(input_size=input_size)
results = nn.train_network(train_inputs, train_targets, test_inputs, test_targets, expected_signals)
print("Melhores Métricas de Teste:", results[-3])  # Imprime os melhores R2 e MSE
print("Análise Final da Correspondência de Ganhos:", results[-1])  # Imprime os resultados finais da correspondência de ganhos para o melhor modelo

### Análise da Performance do Método 2 com Restrição dos Sinais dos Ganhos

In [ ]:
# Finalizando e exibindo contagem de tempo de processamento:
fim = time.time()
print("Tempo de processamento (s):", fim-ini)

# Guardando Métricas:
mse_train2_cons = results[0]
mse_test2_cons = results[1]
r2_train2_cons = results[2]
r2_test2_cons = results[3]
best_model2_cons_params = results[5]

In [ ]:
# Exibindo, em tela, R2 máximo de teste:
print("R2 máximo de teste obtido:", max(r2_test2_cons))

# Exibindo, em tela, número ótimo de neurônios ocultos:
print("Número ótimo de neurônios ocultos:", (r2_test2_cons.index(max(r2_test2_cons))) + 1)

# Análise final de desempenho do Método 2 com restrições de ganhos
print("MSE de teste final com o melhor modelo:", best_metrics['mse'])

In [ ]:
# Criação e exibição de gráfico do MSE de treino:
plt.plot(mse_train2_cons)
plt.ylabel('MSE de Treinamento')
plt.xlabel('Neurônios Ocultos')
plt.legend(['Constrained RIXM'])
plt.show()

In [ ]:
# Criação e exibição de Gráfico do R2 de treino:
#plt.plot(r2_train1_cons)
#plt.ylabel('Train R2')
#plt.xlabel('Hidden Nodes')
#plt.legend(['Constrained WILCAR'])
#plt.show()

In [ ]:
# Criação e exibição de gráfico do MSE de teste:
plt.plot(mse_test2_cons)
plt.ylabel('MSE de Teste')
plt.xlabel('Neurônios Ocultos')
plt.legend(['Constrained RIXM'])
plt.show()

In [ ]:
# Criação e exibição de gráfico do R2 de teste:
plt.plot(r2_test2_cons)
plt.ylabel('R2 de Teste')
plt.xlabel('Neurônios Ocultos')
plt.legend(['Constrained RIXM'])
plt.show()

### Salvando Métricas do Método 2 com Restrições nos Sinais dos Ganhos (Constrained RIXM)

In [ ]:
# Salvando métricas do Método 2 com restrições nos sinais dos ganhos (Constrained RIXM)
metrics_2_cons = {
    'mse_train': mse_train2_cons,
    'mse_test': mse_test2_cons,
    'r2_test': r2_test2_cons
}

In [ ]:
save_metrics(metrics_2_cons, 'Constrained_RIXM')

## Método 3 (ELM)
Modelo ELM considerando os pesos iniciais e bias do neurônio oculto e do neurônio de saída obtidos a partir do método de inicialização randômica clássica, desconsiderando os pesos já aprendidos.

### Definição do Método 3

In [ ]:
# Função de ativação ReLU:
def relu(x):
    return np.maximum(x, 0)

# Função para calcular nós ocultos:
def hidden_nodes(X, input_weights, biases):
    G = np.dot(X, input_weights)
    G = G + biases
    H = relu(G)
    return H

# Função de previsão:
def predict_test(X, output_weights, input_weights, biases):
    out = hidden_nodes(X, input_weights, biases)
    out = np.dot(out, output_weights)
    return out

# Função para cálculo de ganhos:
def calculate_gains_elm(input_weights, biases, output_weights, x_base, delta=0.1):
    n_inputs = x_base.shape[0]
    gains_calculated = []

    for i in range(n_inputs):
        x_perturbed = np.copy(x_base)
        x_perturbed[i, :] += delta  # Perturba cada variável de entrada individualmente

        # Previsão base
        out_base = hidden_nodes(x_base.T, input_weights, biases)
        y_base = np.dot(out_base, output_weights)

        # Previsão perturbada
        out_perturbed = hidden_nodes(x_perturbed.T, input_weights, biases)
        y_perturbed = np.dot(out_perturbed, output_weights)

        gain_i = (y_perturbed - y_base) / delta
        gains_calculated.append(np.mean(gain_i))  # Calcula a média dos ganhos

    return gains_calculated

### Implementação do Método 3

In [ ]:
# Inicializando contagem de tempo:
ini = time.time()

# Abrindo listas para R2 (treino e teste) e MSE:
mse_train_3 = []
mse_test_3 = []
r2_train_3 = []
r2_test_3 = []

# Abrindo lista para registro de pesos dos modelos:
params_models_3 = []

# Definindo tamanho do conjunto de entrada:
input_size = train_inputs.shape[1]

# Inicializando loop para neurônios ocultos:
for n in range(1, 251):
    # Treino:
    input_weights = np.random.uniform(-1, 1, (input_size, n))
    biases = np.zeros(n)
    print("Iniciando rede com", n, "neurônios ocultos")
    out = hidden_nodes(train_inputs, input_weights, biases)
    output_weights = np.dot(pinv(out), train_targets)
    out = np.dot(out, output_weights)
    actual = train_targets
    mse_train_3.append(mse(actual, out))
    r2_train_3.append(r2_score(actual, out))
    # Registrando pesos já aprendidos:
    params_models_3.append((input_weights, biases, output_weights))

    # Teste:
    test_predict = predict_test(test_inputs, output_weights, input_weights, biases)
    actual = test_targets
    mse_test_3.append(mse(actual, test_predict))
    r2_test_3.append(r2_score(actual, test_predict))

    x_base = test_inputs.T

    # Calcula ganhos:
    delta = 0.1
    gains_calculated = calculate_gains_elm(input_weights, biases, output_weights, x_base, delta)

    # Compara ganhos calculados com sinais esperados:
    for i, (gain_calculated, expected_signal) in enumerate(zip(gains_calculated, expected_signals)):
        calculated_signal = np.sign(gain_calculated)
        if expected_signal == 0:
            print(f"Variável {i+1}: Ganho calculado = {calculated_signal}, Ganho esperado = Desconhecido")
        elif calculated_signal == expected_signal:
            print(f"Variável {i+1}: Ganho calculado = {calculated_signal}, Ganho esperado = {expected_signal} [Conforme]")
        else:
            print(f"Variável {i+1}: Ganho calculado = {calculated_signal}, Ganho esperado = {expected_signal} [Divergente]")

# Análise da Performance do Método 3:
fim = time.time()
print("Tempo de processamento:", fim-ini)


### Avaliação do Método 3

In [ ]:
# Verificação de ganhos no melhor modelo (usando o modelo com melhor R2 no teste):
best_model_idx = np.argmax(r2_test_3)
best_input_weights, best_biases, best_output_weights = params_models_3[best_model_idx]

x_base = test_inputs.T

# Calcula ganhos:
delta = 0.1
gains_calculated = calculate_gains_elm(best_input_weights, best_biases, best_output_weights, x_base, delta)
print('ANÁLISE DE GANHOS - MELHOR MODELO - ELM')
# Compara ganhos calculados com sinais esperados:
for i, (gain_calculated, expected_signal) in enumerate(zip(gains_calculated, expected_signals)):
    calculated_signal = np.sign(gain_calculated)
    if expected_signal == 0:
        print(f"Variável {i+1}: Ganho calculado = {calculated_signal}, Ganho esperado = Desconhecido")
    elif calculated_signal == expected_signal:
        print(f"Variável {i+1}: Ganho calculado = {calculated_signal}, Ganho esperado = {expected_signal} [Conforme]")
    else:
        print(f"Variável {i+1}: Ganho calculado = {calculated_signal}, Ganho esperado = {expected_signal} [Divergente]")

In [ ]:
# Criação e exibição de Gráfico do MSE de treino:
plt.plot(mse_train_3)
plt.ylabel('MSE de Treinamento')
plt.xlabel('Neurônios Ocultos')
plt.legend(['ELM'])
plt.show()

In [ ]:
# Criação e exibição de Gráfico do MSE de teste:
plt.plot(mse_test_3)
plt.ylabel('MSE de Teste')
plt.xlabel('Neurônios Ocultos')
plt.legend(['ELM'])
plt.show()

In [ ]:
# Criação e exibição de Gráfico do R2 de treino:
#plt.plot(r2_train_3)
#plt.ylabel('Train R2')
#plt.xlabel('Hidden Nodes')
#plt.legend(['ELM'])
#plt.show()
## Exibindo em tela R2 de treino:
#print("R2 treino associado ao R2 máximo de teste:",
#      r2_train_3[r2_test_3.index(max(r2_test_3))])

In [ ]:
# Criação e exibição de Gráfico do R2 de teste:
plt.plot(r2_test_3)
plt.ylabel('R2 de Teste')
plt.xlabel('Neurônios Ocultos')
plt.legend(['ELM'])
plt.show()

In [ ]:
# Exibindo em tela R2 máximo de teste:
print("R2 máximo de teste obtido:", max(r2_test_3))

# Exibindo em tela número ótimo de neurônios ocultos:
print("Número ótimo de neurônios ocultos:", (r2_test_3.index(max(r2_test_3))) + 1)

### Salvando Métricas do Método 3 (ELM)

In [ ]:
### Salvando Métricas do Método 3 (ELM)
metrics_3 = {
    'mse_train': mse_train_3,
    'mse_test': mse_test_3,
    'r2_test': r2_test_3
}

In [ ]:
save_metrics(metrics_3, 'ELM')

## Método 3 (ELM) - Com Restrições nos Sinais dos Ganhos

### Definição do Método 3 com Restrições nos Sinais dos Ganhos

In [ ]:
# Função de ativação ReLU
def relu(x):
    return np.maximum(x, 0)

# Função para calcular nós ocultos
def hidden_nodes(X, input_weights, biases):
    G = np.dot(X, input_weights)
    G = G + biases
    H = relu(G)
    return H

# Função de previsão
def predict_test(X, output_weights, input_weights, biases):
    out = hidden_nodes(X, input_weights, biases)
    out = np.dot(out, output_weights)
    return out

# Função de restrição de ganho
def gain_constraint_elm(x, idx, x_base, expected_signals, delta=0.1, input_size=6, hidden_neurons=1):
    input_weights = x[:input_size * hidden_neurons].reshape(input_size, hidden_neurons)
    biases = x[input_size * hidden_neurons: input_size * hidden_neurons + hidden_neurons]
    output_weights = x[input_size * hidden_neurons + hidden_neurons:]
    
    x_var = x_base.copy()
    x_var[idx] += delta
    y_base = predict_test(x_base.T, output_weights, input_weights, biases)
    new_y = predict_test(x_var.T, output_weights, input_weights, biases)
    gain = (new_y - y_base) / delta
    
    if expected_signals[idx] != 0:
        return expected_signals[idx] * gain - 0.01
    else:
        return 1

# Função para cálculo de ganhos:
def calculate_gains_elm(input_weights, biases, output_weights, x_base, delta=0.1):
    n_inputs = x_base.shape[0]
    gains_calculated = []

    for i in range(n_inputs):
        x_perturbed = np.copy(x_base)
        x_perturbed[i, :] += delta  # Perturba cada variável de entrada individualmente

        # Previsão base
        out_base = hidden_nodes(x_base.T, input_weights, biases)
        y_base = np.dot(out_base, output_weights)

        # Previsão perturbada
        out_perturbed = hidden_nodes(x_perturbed.T, input_weights, biases)
        y_perturbed = np.dot(out_perturbed, output_weights)

        gain_i = (y_perturbed - y_base) / delta
        gains_calculated.append(np.mean(gain_i))  # Calcula a média dos ganhos

    return gains_calculated

def train_elm_with_constraints(W0, cons, train_inputs, train_targets, hidden_neurons, input_size):
    def objective_function_elm(W):
        input_weights = W[:input_size * hidden_neurons].reshape(input_size, hidden_neurons)
        biases = W[input_size * hidden_neurons: input_size * hidden_neurons + hidden_neurons]
        output_weights = W[input_size * hidden_neurons + hidden_neurons:]
        H = hidden_nodes(train_inputs, input_weights, biases)
        predictions = np.dot(H, output_weights)
        regularization = 0.01 * (np.sum(np.square(input_weights)) + np.sum(np.square(output_weights)))
        return mse(train_targets, predictions) + regularization

    solution = minimize(objective_function_elm, W0, method='SLSQP', constraints=cons, options={'disp': True, 'maxiter': 500, 'ftol': 1e-06})
    return solution['x']

def initialize_weights_with_constraints(input_size, hidden_neurons, expected_signals, x_base, delta=0.1):
    def xavier_initialization(input_size, hidden_neurons):
        limit = np.sqrt(6 / (input_size + hidden_neurons))
        W0 = np.random.uniform(-limit, limit, (input_size * hidden_neurons + hidden_neurons + hidden_neurons))
        return W0

    first_attempt = True
    while True:
        W0 = xavier_initialization(input_size, hidden_neurons)
        input_weights = W0[:input_size * hidden_neurons].reshape(input_size, hidden_neurons)
        biases = W0[input_size * hidden_neurons: input_size * hidden_neurons + hidden_neurons]
        output_weights = W0[input_size * hidden_neurons + hidden_neurons:]

        gains_calculated = calculate_gains_elm(input_weights, biases, output_weights, x_base, delta)

        all_gains_match = True
        for i, expected_signal in enumerate(expected_signals):
            if expected_signal != 0:
                calculated_signal = np.sign(gains_calculated[i])
                if calculated_signal != expected_signal:
                    all_gains_match = False
                    break
        
        if all_gains_match:
            return W0

        if first_attempt:
            print("Pesos iniciais não satisfazem as restrições, tentando novamente.")
            first_attempt = False

### Implementação do Método 3 com Restrições nos Sinais dos Ganhos

In [ ]:
# Inicializar variáveis
mse_train_3_cons = []
mse_test_3_cons = []
r2_train_3_cons = []
r2_test_3_cons = []
params_models_3_cons = []

# Definir tamanho da entrada
input_size = train_inputs.shape[1]

# Inicializar temporizador
ini = time.time()

# Loop sobre neurônios ocultos
for n in range(1, 251):
    print(f"Iniciando rede com {n} neurônios ocultos")

    x_base = np.mean(test_inputs, axis=0).reshape(input_size, 1)
    
    while True:
        W0 = initialize_weights_with_constraints(input_size, n, expected_signals, x_base)
        cons = [{'type': 'ineq', 'fun': lambda W, idx=i: gain_constraint_elm(W, idx, x_base, expected_signals, delta=0.1, input_size=input_size, hidden_neurons=n)} for i in range(input_size)]
        
        W_opt = train_elm_with_constraints(W0, cons, train_inputs, train_targets, n, input_size)
        
        input_weights = W_opt[:input_size * n].reshape(input_size, n)
        biases = W_opt[input_size * n:input_size * n + n]
        output_weights = W_opt[input_size * n + n:]
        
        # Calcula ganhos:
        gains_calculated = calculate_gains_elm(input_weights, biases, output_weights, test_inputs.T)

        # Verifica correspondência de ganhos
        all_gains_match = True
        for i, (gain_calculated, expected_signal) in enumerate(zip(gains_calculated, expected_signals)):
            calculated_signal = np.sign(gain_calculated)
            if expected_signal == 0:
                print(f"Variável {i+1}: Ganho calculado = {calculated_signal}, Ganho esperado = Desconhecido")
            elif calculated_signal == expected_signal:
                print(f"Variável {i+1}: Ganho calculado = {calculated_signal}, Ganho esperado = {expected_signal} [Conforme]")
            else:
                print(f"Variável {i+1}: Ganho calculado = {calculated_signal}, Ganho esperado = {expected_signal} [Divergente]")
                all_gains_match = False

        if all_gains_match:
            break
        else:
            print("Pesos após adição do neurônio não satisfazem as restrições, recalculando...")

    # Treinar
    H = hidden_nodes(train_inputs, input_weights, biases)
    out_train = np.dot(H, output_weights)
    mse_train_3_cons.append(mse(train_targets, out_train))
    r2_train_3_cons.append(r2_score(train_targets, out_train))
    params_models_3_cons.append((input_weights, biases, output_weights))
    
    # Testar
    out_test = predict_test(test_inputs, output_weights, input_weights, biases)
    mse_test_3_cons.append(mse(test_targets, out_test))
    r2_test_3_cons.append(r2_score(test_targets, out_test))

    # Calcula ganhos:
    gains_calculated = calculate_gains_elm(input_weights, biases, output_weights, test_inputs.T)

    # Verifica correspondência de ganhos
    all_gains_match = True
    for i, (gain_calculated, expected_signal) in enumerate(zip(gains_calculated, expected_signals)):
        calculated_signal = np.sign(gain_calculated)
        if expected_signal == 0:
            print(f"Variável {i+1}: Ganho calculado = {calculated_signal}, Ganho esperado = Desconhecido")
        elif calculated_signal == expected_signal:
            print(f"Variável {i+1}: Ganho calculado = {calculated_signal}, Ganho esperado = {expected_signal} [Conforme]")
        else:
            print(f"Variável {i+1}: Ganho calculado = {calculated_signal}, Ganho esperado = {expected_signal} [Divergente]")
            all_gains_match = False

    if not all_gains_match:
        print(f"Treinamento interrompido em {n} neurônios. Problema de correspondência de ganhos.")
        break

# Análise da Performance do Método 3:
fim = time.time()
print("Tempo de processamento:", fim-ini)

### Avaliação do Método 3 com Restrições nos Sinais dos Ganhos

In [ ]:
# Verificação de ganhos no melhor modelo
best_model_idx = np.argmax(r2_test_3_cons)
best_input_weights, best_biases, best_output_weights = params_models_3_cons[best_model_idx]

x_base = test_inputs.T

# Calcular ganhos
delta = 0.1
gains_calculated = calculate_gains_elm(best_input_weights, best_biases, best_output_weights, x_base, delta)

# Comparar ganhos calculados com sinais esperados
for i, (gain_calculated, expected_signal) in enumerate(zip(gains_calculated, expected_signals)):
    calculated_signal = np.sign(gain_calculated)
    if expected_signal == 0:
        print(f"Variável {i+1}: Ganho calculado = {calculated_signal}, Ganho esperado = Desconhecido")
    elif calculated_signal == expected_signal:
        print(f"Variável {i+1}: Ganho calculado = {calculated_signal}, Ganho esperado = {expected_signal} [Conforme]")
    else:
        print(f"Variável {i+1}: Ganho calculado = {calculated_signal}, Ganho esperado = {expected_signal} [Divergente]")


In [ ]:
# Criação e exibição de Gráfico do MSE de treino:
plt.plot(mse_train_3_cons)
plt.ylabel('Train MSE')
plt.xlabel('Hidden Nodes')
plt.legend(['Constrained ELM'])
plt.show()

In [ ]:
# Criação e exibição de Gráfico do MSE de treino:
plt.plot(mse_test_3_cons)
plt.ylabel('Test MSE')
plt.xlabel('Hidden Nodes')
plt.legend(['Constrained ELM'])
plt.show()

In [ ]:
# Criação e exibição de Gráfico do R2 de treino:
#plt.plot(r2_train_3_cons)
#plt.ylabel('Train R2')
#plt.xlabel('Hidden Nodes')
#plt.legend(['Constrained ELM'])
#plt.show()
## Exibindo em tela R2 de treino:
#print("R2 treino associado ao R2 máximo de teste:",
#      r2_train_3_cons[r2_test_3_cons.index(max(r2_test_3_cons))])

In [ ]:
# Criação e exibição de Gráfico do R2 de teste:
plt.plot(r2_test_3_cons)
plt.ylabel('Test R2')
plt.xlabel('Hidden Nodes')
plt.legend(['Constrained ELM'])
plt.show()

In [ ]:
print("R2 máximo de teste obtido:", max(r2_test_3_cons))
print("Número ótimo de neurônios ocultos:", (r2_test_3_cons.index(max(r2_test_3_cons))) + 1)

### Salvando Métricas do Método 3 com Restrições nos Sinais dos Ganhos (Constrained ELM)

In [ ]:
metrics_3_cons = {
    'mse_train': mse_train_3_cons,
    'mse_test': mse_test_3_cons,
    'r2_test': r2_test_3_cons
}

In [ ]:
save_metrics(metrics_3_cons, 'Constrained ELM')

## GRÁFICOS DOS MÉTODOS EM CONJUNTO

In [ ]:
import matplotlib
matplotlib.rc('xtick', labelsize=10)
matplotlib.rc('ytick', labelsize=10)

### MSE de Treino

In [ ]:
# Gráfico em conjunto do MSE de treino (até 250 neurônios ocultos)
x = np.arange(1, 251)
y1 = mse_train_1
y2 = mse_train1_cons_mse
y3 = mse_train1_cons_rmse
y4 = mse_train_2
y5 = mse_train2_cons
y6 = mse_train_3
y7 = mse_train_3_cons

# Inicializa a figura e os eixos
fig, ax = plt.subplots(1, figsize=(12, 8), dpi=100)

# Plota todas as linhas no mesmo gráfico, atribuindo um rótulo para cada uma para ser exibido na legenda
ax.plot(x, y1, color="green", label="WILCAR")
ax.plot(x, y2, label="Constrained WILCAR (MSE as obj. fnc.)")
ax.plot(x, y3, label="Constrained WILCAR (RMSE as obj. fnc.)")
ax.plot(x, y4, color="red", label="RIXM")
ax.plot(x, y5, label="Constrained RIXM")
ax.plot(x, y6, color="blue", label="ELM")
ax.plot(x, y7, label="Constrained ELM")

# Adiciona uma legenda e a posiciona no canto inferior direito (sem caixa)
plt.legend()

# Nome dos eixos
plt.ylabel('MSE (amostra de treino)', fontsize=12)
plt.xlabel('Número de unidades ocultas', fontsize=12)
# Para ampliar o gráfico
# plt.xlim(0, 50)
plt.show()

### R2 de Teste

In [ ]:
# Gráfico em conjunto do R2 de teste (até 250 neurônios ocultos)
x = np.arange(1, 251)
y1 = r2_test_1
y2 = r2_test1_cons_mse
y3 = r2_test1_cons_rmse
y4 = r2_test_2
y5 = r2_test2_cons
y6 = r2_test_3
y7 = r2_test_3_cons

# Inicializa a figura e os eixos
fig, ax = plt.subplots(1, figsize=(12, 8), dpi=100)

# Plota todas as linhas no mesmo gráfico, atribuindo um rótulo para cada uma para ser exibido na legenda
ax.plot(x, y1, color="green", label="WILCAR")
ax.plot(x, y2, label="Constrained WILCAR (MSE as obj. fnc.)")
ax.plot(x, y3, label="Constrained WILCAR (RMSE as obj. fnc.)")
ax.plot(x, y4, color="red", label="RIXM")
ax.plot(x, y5, label="Constrained RIXM")
ax.plot(x, y6, color="blue", label="ELM")
ax.plot(x, y7, label="Constrained ELM")

# Adiciona uma legenda e a posiciona no canto inferior direito (sem caixa)
plt.legend()

# Nome dos eixos
plt.ylabel('R2 (amostra de teste)', fontsize=12)
plt.xlabel('Número de unidades ocultas', fontsize=12)
plt.ylim(0, 1)
plt.show()

## FIM DO CÓDIGO